# HostWise — Shared Modeling Foundation (EUR)

Creates the **single shared train/test split** and the **shared regression target** used by all modeling work.

- **Population:** five cities, current listings (`city scrape`), valid price, `minimum_nights <= 29`
- **Target:** `price_eur`, the advertised nightly price in EUR. London prices are converted from GBP with a fixed approximate rate (section 4); the local-currency price `price_num` is kept unchanged.
- **Shared dataset:** all original columns are kept, so every feature set uses the same training and test rows.
- **Split:** 80/20, stratified by city, `random_state=42`
- **Outputs** (`../data/splits/`): `train_ids_eur.csv`, `test_ids_eur.csv`, `split_config_eur.json`

No models are trained, and the test set is not used for any decision.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.float_format", "{:,.2f}".format)

# Paths
DATA_PATH = Path("../data/processed/hostwise_5cities.csv")
SPLIT_DIR = Path("../data/splits")
NEW_TRAIN_IDS = SPLIT_DIR / "train_ids_eur.csv"
NEW_TEST_IDS = SPLIT_DIR / "test_ids_eur.csv"
NEW_CONFIG = SPLIT_DIR / "split_config_eur.json"

# Agreed settings
CITIES = ["london", "paris", "rome", "madrid", "barcelona"]
MAX_MIN_NIGHTS = 29
TEST_SIZE = 0.20
RANDOM_STATE = 42

LOCAL_PRICE = "price_num"   # original nightly price, local currency (kept unchanged)
TARGET = "price_eur"        # shared regression target, EUR

print("Data file found:", DATA_PATH.exists())

Data file found: True


## 1. Load and validate the dataset
Check that the file contains exactly the five project cities and that every listing ID is unique. IDs are read as 64-bit integers so the long Airbnb IDs are never rounded. The file's SHA-256 fingerprint is recorded so teammates can confirm they use the same dataset version.

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False, dtype={"id": "int64"})

assert set(df["city"].unique()) == set(CITIES), f"Unexpected cities: {sorted(df['city'].unique())}"
assert df["id"].is_unique, f"{df['id'].duplicated().sum()} duplicate listing IDs"

DATASET_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]} | SHA-256: {DATASET_SHA256[:16]}…")
df["city"].value_counts().reindex(CITIES).to_frame("listings")

Rows: 245,402 | Columns: 92 | SHA-256: ea79f802b2ffdb37…


,listings
city,
london,92638
paris,77679
rome,37084
madrid,22708
barcelona,15293


## 2. Local price (`price_num`)
`price` is text such as "$1,234.00". The `$` is only formatting: prices are in local currency (London GBP, other cities EUR). It is parsed into `price_num`, which stays in local currency throughout. A valid price is present, parseable and greater than 0.

In [3]:
df[LOCAL_PRICE] = pd.to_numeric(
    df["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce"
)

is_active = df["source"] == "city scrape"

pd.Series({
    "active listings": is_active.sum(),
    "price missing": (is_active & df["price"].isna()).sum(),
    "price present but not parseable": (is_active & df["price"].notna() & df[LOCAL_PRICE].isna()).sum(),
    "price <= 0": (is_active & (df[LOCAL_PRICE] <= 0)).sum(),
    "valid price": (is_active & (df[LOCAL_PRICE] > 0)).sum(),
}).to_frame("listings")

,listings
active listings,174922
price missing,6197
price present but not parseable,0
price <= 0,0
valid price,168725


## 3. Modeling scope
Three cumulative conditions:
1. `source == "city scrape"`: listings in the current scrape
2. valid price (> 0)
3. `minimum_nights <= 29`: short stays. Rows with missing `minimum_nights` are excluded, as in Day 1.

No other filters are applied. Outlier rules, room-type exclusions and missing-value handling are not part of this step.

In [4]:
scope_conditions = {
    "active (city scrape)": is_active,
    "+ valid price": df[LOCAL_PRICE] > 0,
    f"+ minimum_nights <= {MAX_MIN_NIGHTS}": df["minimum_nights"] <= MAX_MIN_NIGHTS,
}

in_scope = pd.Series(True, index=df.index)
scope_steps = {"all listings": df["city"].value_counts()}
for step, condition in scope_conditions.items():
    in_scope &= condition
    scope_steps[step] = df.loc[in_scope, "city"].value_counts()

df_model = df.loc[in_scope].copy()

scope_table = pd.DataFrame(scope_steps).reindex(CITIES).T
scope_table["total"] = scope_table.sum(axis=1)
scope_table

city,london,paris,rome,madrid,barcelona,total
all listings,92638,77679,37084,22708,15293,245402
active (city scrape),61587,48140,33612,18548,13035,174922
+ valid price,59345,45215,33136,18263,12766,168725
+ minimum_nights <= 29,58810,39369,31275,16257,8101,153812


## 4. Currency standardization
All prices are expressed in **EUR**. Paris, Rome, Madrid and Barcelona are already in EUR and stay unchanged. London prices are converted from GBP with a **fixed rate of 1 GBP = 1.15 EUR**.

**Assumption and limitations**
- The rate is a documented **approximation**, not an exact historical rate. For reference, the ECB's daily reference rates in June 2026 correspond to about 1.154–1.161 EUR per GBP, so 1.15 is up to about 1% lower.
- One rate is applied to all London listings, regardless of scrape date. No rows are filtered by date.
- The conversion changes the currency unit only, not price levels or purchasing power. London remains a different market from the euro cities.
- `price_num` is kept unchanged in local currency for traceability.

In [5]:
GBP_TO_EUR = 1.15   # fixed approximate conversion rate (documented assumption)

df_model[TARGET] = df_model[LOCAL_PRICE] * np.where(df_model["city"] == "london", GBP_TO_EUR, 1.0)

In [6]:
assert df_model[TARGET].notna().all() and (df_model[TARGET] > 0).all(), "Invalid price_eur values"
is_eur_city = df_model["city"] != "london"
assert (df_model.loc[is_eur_city, TARGET] == df_model.loc[is_eur_city, LOCAL_PRICE]).all(), "EUR cities changed"

fx_check = df_model.groupby("city")[[LOCAL_PRICE, TARGET]].median().reindex(CITIES)
fx_check["ratio"] = fx_check[TARGET] / fx_check[LOCAL_PRICE]
fx_check

,price_num,price_eur,ratio
city,,,
london,183.43,210.94,1.15
paris,230.00,230.00,1.00
rome,166.33,166.33,1.00
madrid,131.00,131.00,1.00
barcelona,254.86,254.86,1.00


## 5. Shared dataset and baseline features
The split is applied to the full `df_model`: all original columns plus `price_num` and `price_eur`. Later feature sets can select any columns from exactly the same rows without a new split.

The eight agreed **baseline features** are defined here for reference. A check confirms that they exist and that none of them is a target-derived or leakage column.

In [7]:
NUMERIC_FEATURES = ["accommodates", "bedrooms", "bathrooms", "latitude", "longitude"]
CATEGORICAL_FEATURES = ["city", "room_type", "property_type"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

LEAKAGE_COLUMNS = [
    "price", LOCAL_PRICE, TARGET, "estimated_revenue_l365d",
    "price_quote_total_price", "price_quote_price_per_night", "price_quote_raw",
    "price_quote_checkin_date", "price_quote_checkout_date",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "availability_eoy", "estimated_occupancy_l365d",
]

assert set(FEATURES) <= set(df_model.columns), "A baseline feature is missing from the dataset"
assert not set(FEATURES) & set(LEAKAGE_COLUMNS), "A leakage column is in the baseline features"

pd.DataFrame({
    "dtype": df_model[FEATURES + [TARGET]].dtypes.astype(str),
    "missing_%": df_model[FEATURES + [TARGET]].isna().mean() * 100,
})

,dtype,missing_%
accommodates,int64,0.00
bedrooms,float64,17.78
bathrooms,float64,6.35
latitude,float64,0.00
longitude,float64,0.00
city,object,0.00
room_type,object,0.00
property_type,object,0.00
price_eur,float64,0.00


## 6. Train/test split
Random 80/20 split of the full modeling dataset, stratified by `city`, `random_state=42`.
Rows are sorted by `id` first, because `train_test_split` depends on row order; sorting makes the split independent of how the CSV is ordered.
The split keeps **all columns**, so every feature set uses exactly the same training and test rows.
Limitation: the split is at listing level, so one host's listings can appear in both train and test (measured in section 7).

In [8]:
df_model = df_model.sort_values("id").reset_index(drop=True)

train_df, test_df = train_test_split(
    df_model,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_model["city"],
)

train_ids, test_ids = train_df["id"], test_df["id"]
print(f"train: {train_df.shape} | test: {test_df.shape}")

train: (123049, 93) | test: (30763, 93)


## 7. Verify the split
Train and test must not share IDs and must together cover the full modeling dataset. City shares should match in train and test. Host overlap is reported for information only.

In [9]:
assert train_ids.is_unique and test_ids.is_unique, "Duplicate IDs inside train or test"
assert set(train_ids).isdisjoint(test_ids), "Train and test share listing IDs"
assert set(train_ids) | set(test_ids) == set(df_model["id"]), "Split does not cover the full dataset"

pd.DataFrame({
    "scope_%": df_model["city"].value_counts(normalize=True) * 100,
    "train_%": train_df["city"].value_counts(normalize=True) * 100,
    "test_%": test_df["city"].value_counts(normalize=True) * 100,
    "train_n": train_df["city"].value_counts(),
    "test_n": test_df["city"].value_counts(),
}).reindex(CITIES)

,scope_%,train_%,test_%,train_n,test_n
city,,,,,
london,38.23,38.24,38.23,47048,11762
paris,25.60,25.60,25.60,31495,7874
rome,20.33,20.33,20.33,25020,6255
madrid,10.57,10.57,10.57,13005,3252
barcelona,5.27,5.27,5.27,6481,1620


In [10]:
train_hosts = set(train_df["host_id"])
shared_host = test_df["host_id"].isin(train_hosts)

print(f"Test listings whose host also has a listing in train: {shared_host.mean():.1%}")

Test listings whose host also has a listing in train: 55.8%


## 8. Save the shared split
Saves the train and test listing IDs plus a short configuration file documenting the dataset version, scope, target, conversion rate and split settings. If the files already exist, they are not overwritten. The run only checks that the split is identical.

In [11]:
split_config = {
    "dataset": {"file": DATA_PATH.name, "sha256": DATASET_SHA256, "rows_loaded": len(df)},
    "scope": list(scope_conditions),
    "scope_rows": len(df_model),
    "target": {"name": TARGET, "local_price": LOCAL_PRICE, "gbp_to_eur": GBP_TO_EUR,
               "note": "fixed approximate rate; London converted, other cities unchanged"},
    "baseline_features": FEATURES,
    "split": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE, "stratify": "city", "rows_sorted_by": "id"},
    "n_train": len(train_ids),
    "n_test": len(test_ids),
    "test_listings_with_host_in_train_%": round(shared_host.mean() * 100, 1),
}

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

if NEW_CONFIG.exists():
    saved_train = pd.read_csv(NEW_TRAIN_IDS, dtype={"id": "int64"})["id"]
    saved_test = pd.read_csv(NEW_TEST_IDS, dtype={"id": "int64"})["id"]
    assert set(saved_train) == set(train_ids) and set(saved_test) == set(test_ids), \
        "Saved split differs from this run - NOT overwritten. Investigate before changing anything."
    print("Split files already exist and are identical. Nothing overwritten.")
else:
    train_ids.to_frame().to_csv(NEW_TRAIN_IDS, index=False)
    test_ids.to_frame().to_csv(NEW_TEST_IDS, index=False)
    NEW_CONFIG.write_text(json.dumps(split_config, indent=2))
    print(f"Saved {NEW_TRAIN_IDS.name}, {NEW_TEST_IDS.name} and {NEW_CONFIG.name} to {SPLIT_DIR}")

Split files already exist and are identical. Nothing overwritten.


## 9. Reload check
Rebuilds the train and test data from the saved files, exactly as teammates will, and confirms the result is identical to the split created above.

In [12]:
saved_config = json.loads(NEW_CONFIG.read_text())
saved_train_ids = pd.read_csv(NEW_TRAIN_IDS, dtype={"id": "int64"})["id"]
saved_test_ids = pd.read_csv(NEW_TEST_IDS, dtype={"id": "int64"})["id"]

assert saved_config["dataset"]["sha256"] == DATASET_SHA256, "Different dataset version"
assert saved_config["target"]["gbp_to_eur"] == GBP_TO_EUR, "Different conversion rate"

indexed = df_model.set_index("id")
train_reloaded, test_reloaded = indexed.loc[saved_train_ids], indexed.loc[saved_test_ids]

assert train_reloaded.sort_index().equals(train_df.set_index("id").sort_index())
assert test_reloaded.sort_index().equals(test_df.set_index("id").sort_index())
print(f"Reload OK: train {train_reloaded.shape}, test {test_reloaded.shape}, target = {saved_config['target']['name']}")

Reload OK: train (123049, 92), test (30763, 92), target = price_eur


## 10. How teammates use this foundation

Every notebook rebuilds the shared data from the saved IDs. No notebook creates its own split.

```python
DATA_PATH = Path("../data/processed/hostwise_5cities.csv")
SPLIT_DIR = Path("../data/splits")

df = pd.read_csv(DATA_PATH, low_memory=False, dtype={"id": "int64"}).set_index("id")
df["price_num"] = pd.to_numeric(df["price"].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
df["price_eur"] = df["price_num"] * np.where(df["city"] == "london", 1.15, 1.0)

train_ids = pd.read_csv(SPLIT_DIR / "train_ids_eur.csv", dtype={"id": "int64"})["id"]
test_ids = pd.read_csv(SPLIT_DIR / "test_ids_eur.csv", dtype={"id": "int64"})["id"]
train_df, test_df = df.loc[train_ids], df.loc[test_ids]

# any feature set, same rows:
X_train, y_train = train_df[FEATURES], train_df["price_eur"]
X_test, y_test = test_df[FEATURES], test_df["price_eur"]
```

**Rules**
- The saved IDs already contain only in-scope listings, so the scope filters are not repeated.
- Target is always `price_eur`; the conversion rate is 1.15 (see `split_config_eur.json`).
- Select any columns for a feature set, but never columns from the leakage list in section 5.
- Fit all preprocessing on training data or training folds only; use the test set once, for the final evaluation.

## Summary
- **Population:** 153,812 listings (five cities, current listings, valid price, `minimum_nights <= 29`).
- **Target:** `price_eur`. London converted at a fixed 1 GBP = 1.15 EUR (approximation); other cities unchanged; `price_num` kept.
- **Split:** 123,049 train / 30,763 test, stratified by city, `random_state=42`, on the full dataset with all columns.
- **Saved:** `train_ids_eur.csv`, `test_ids_eur.csv`, `split_config_eur.json` in `../data/splits/`.
- **Caveat:** about 56% of test listings have a host who also appears in training. Report this with model results.